# 03 — Model Training & Comparison

**Student Academic Risk — Early Intervention System**

This notebook trains and compares multiple ML models:
1. Logistic Regression (baseline)
2. Decision Tree
3. Random Forest
4. XGBoost
5. Support Vector Machine (SVM)

**Target**: Binary classification — LOW vs AT_RISK (MEDIUM + HIGH)

**Key Features**: CA_Score, Attendance_Percentage, Assignment_Average, Test_Average, and engineered features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print('⚠️ XGBoost not installed. Skipping XGBoost model.')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 1. Load Preprocessed Data

In [ ]:
processed_dir = os.path.join('..', 'data_science', 'data', 'processed')
models_dir = os.path.join('..', 'data_science', 'models')

X_train = np.load(os.path.join(processed_dir, 'X_train.npy'))
X_test = np.load(os.path.join(processed_dir, 'X_test.npy'))
y_train = np.load(os.path.join(processed_dir, 'y_train.npy'))
y_test = np.load(os.path.join(processed_dir, 'y_test.npy'))

feature_cols = joblib.load(os.path.join(models_dir, 'feature_columns.joblib'))
le = joblib.load(os.path.join(models_dir, 'label_encoder.joblib'))

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'Classes:      {le.classes_}')
print(f'Train distribution: AT_RISK={sum(y_train==0)}, LOW={sum(y_train==1)}')
print(f'Test distribution:  AT_RISK={sum(y_test==0)}, LOW={sum(y_test==1)}')

## 2. Define Models

In [ ]:
# All models use class_weight='balanced' to handle imbalance
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', max_depth=5, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', max_depth=10, random_state=42
    ),
    'SVM': SVC(
        class_weight='balanced', kernel='rbf', probability=True, random_state=42
    ),
}

if HAS_XGBOOST:
    # Calculate scale_pos_weight for imbalanced classes
    n_negative = sum(y_train == 0)  # AT_RISK (minority)
    n_positive = sum(y_train == 1)  # LOW (majority)
    models['XGBoost'] = XGBClassifier(
        scale_pos_weight=n_negative / n_positive if n_positive > 0 else 1,
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42, eval_metric='logloss'
    )

print(f'Models to train: {list(models.keys())}')

## 3. Train & Evaluate All Models

In [ ]:
results = []
trained_models = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_weighted')
    print(f'Cross-validation F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    
    # Train on full training set
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predict on test set
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else 0
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'CV_F1_Mean': cv_scores.mean(),
        'CV_F1_Std': cv_scores.std()
    })
    
    print(f'\nTest Results:')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'  ROC-AUC:   {auc:.4f}')

print('\n✅ All models trained successfully.')

## 4. Model Comparison Table

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

# Highlight the best model
best_model_name = results_df.iloc[0]['Model']
print(f'🏆 Best Model: {best_model_name} (F1-Score: {results_df.iloc[0]["F1-Score"]:.4f})')
print()

# Display comparison table
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
results_df[display_cols].style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'ROC-AUC': '{:.4f}'
}).highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], 
                color='lightgreen')

## 5. Visual Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(results_df))
width = 0.15
colors = ['#6366F1', '#8B5CF6', '#EC4899', '#F59E0B', '#10B981']

for i, metric in enumerate(metrics):
    axes[0].bar(x + i * width, results_df[metric], width, label=metric, color=colors[i])

axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_ylim(0, 1.1)

# Cross-validation F1 with error bars
axes[1].barh(results_df['Model'], results_df['CV_F1_Mean'],
            xerr=results_df['CV_F1_Std'],
            color='#6366F1', edgecolor='white', capsize=5)
axes[1].set_xlabel('F1-Score (5-Fold CV)')
axes[1].set_title('Cross-Validation F1-Score', fontweight='bold', fontsize=14)
for i, val in enumerate(results_df['CV_F1_Mean']):
    axes[1].text(val + 0.01, i, f'{val:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data_science/data/08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Confusion Matrices

In [ ]:
n_models = len(trained_models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
               xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontweight='bold', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('../data_science/data/09_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Classification Reports

In [ ]:
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    print(f'\n{"="*50}')
    print(f'{name} — Classification Report')
    print(f'{"="*50}')
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

## 8. Feature Importance (Best Model)

In [ ]:
# Get the best model
best_model = trained_models[best_model_name]

# Extract feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
else:
    importances = None
    print('Model does not support feature importance extraction.')

if importances is not None:
    feat_imp = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feat_imp)))
    ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='white')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance — {best_model_name}', fontweight='bold', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('../data_science/data/10_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nTop 5 Most Important Features:')
    for _, row in feat_imp.tail(5).iterrows():
        print(f'  ⭐ {row["Feature"]}: {row["Importance"]:.4f}')

## 9. Save Best Model

In [ ]:
# Save the best model
model_path = os.path.join(models_dir, 'best_model.joblib')
joblib.dump(best_model, model_path)

# Save results table
results_df.to_csv(os.path.join('..', 'data_science', 'data', 'model_comparison_results.csv'), index=False)

# Save model metadata
metadata = {
    'model_name': best_model_name,
    'accuracy': float(results_df.iloc[0]['Accuracy']),
    'precision': float(results_df.iloc[0]['Precision']),
    'recall': float(results_df.iloc[0]['Recall']),
    'f1_score': float(results_df.iloc[0]['F1-Score']),
    'roc_auc': float(results_df.iloc[0]['ROC-AUC']),
    'features': feature_cols,
    'classes': list(le.classes_),
    'target': 'Risk_Binary (AT_RISK vs LOW)'
}
joblib.dump(metadata, os.path.join(models_dir, 'model_metadata.joblib'))

print(f'✅ Best model saved: {model_path}')
print(f'   Model: {best_model_name}')
print(f'   F1-Score: {results_df.iloc[0]["F1-Score"]:.4f}')
print(f'   ROC-AUC: {results_df.iloc[0]["ROC-AUC"]:.4f}')